# Modern Object Detection V: Monocular 3D Detection with Cube R-CNN

3D cuboids from a single RGB image. Inference only — we do not train.


## 0. Workshop introduction

A 2D box says *where on the photo* an object is. A **3D cuboid** also needs:

- depth (how far)
- dimensions (width, height, length in meters)
- orientation (how it is rotated)
- a **camera** model, so pixels correspond to rays in space

Cube R-CNN (Omni3D) predicts those 3D parameters from one RGB image. Depth is inferred, not measured — a single photo does not contain metric depth the way a stereo pair or LiDAR does.


## 1. Learning objectives

- Contrast 2D vs 3D object detection.
- Name the extra quantities needed to lift a 2D box to a cuboid.
- See how **focal length** changes estimated scale/depth.
- Assemble an inference-only 3D pipeline.


## How this workshop is structured

You will **not** implement neural-network layers from scratch.

The instructor cells already contain working functions for each important stage of the algorithm. Your job is to:

1. Read what each stage does and why it exists.
2. Assemble those stages in the correct order (a short coding task).
3. Change one or two parameters and watch the output change.

The demo cell is only a one-liner (`run_full_pipeline`) so you can see a result after Run all. **Do not copy that function for the assembly exercise** — wire the named stages listed in the student task.

Hands-on coding is intentionally light (~20–25% of the session). Most of the time is for understanding the pipeline.


## 2. Environment setup

This notebook is heavier than the others: it installs **Detectron2** and clones [Omni3D](https://github.com/facebookresearch/omni3d). First-time setup can take **5–10 minutes**. We do **not** train Cube R-CNN.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

!pip install -q fvcore iopath yacs cloudpickle omegaconf matplotlib opencv-python-headless pillow scipy pandas
!pip install -q cython pycocotools

# Detectron2 from source matches Colab's current PyTorch. This step is the slow one.
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

if not Path("omni3d").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/facebookresearch/omni3d.git"])

sys.path.insert(0, str(Path("omni3d").resolve()))
print("omni3d on sys.path")


In [ ]:
import platform
import sys

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(props.total_memory / 1024 ** 3, 2), "GB")
else:
    print("GPU: None")
    print("GPU memory: n/a")
    print("\nEnable a GPU: Runtime → Change runtime type → T4 GPU, then Restart session.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import types
import torch.nn.functional as F


def _install_pytorch3d_stub():
    """Minimal transforms used by Cube R-CNN inference if pytorch3d is missing."""
    try:
        from pytorch3d.transforms import rotation_6d_to_matrix, axis_angle_to_matrix  # noqa: F401
        print("pytorch3d transforms available")
        return
    except Exception:
        pass

    def rotation_6d_to_matrix(d6):
        a1, a2 = d6[..., :3], d6[..., 3:]
        b1 = F.normalize(a1, dim=-1)
        b2 = a2 - (b1 * a2).sum(dim=-1, keepdim=True) * b1
        b2 = F.normalize(b2, dim=-1)
        b3 = torch.cross(b1, b2, dim=-1)
        return torch.stack((b1, b2, b3), dim=-2)

    def quaternion_to_matrix(quaternions):
        r, i, j, k = torch.unbind(quaternions, -1)
        two_s = 2.0 / (quaternions * quaternions).sum(-1)
        o = torch.stack(
            (
                1 - two_s * (j * j + k * k),
                two_s * (i * j - k * r),
                two_s * (i * k + j * r),
                two_s * (i * j + k * r),
                1 - two_s * (i * i + k * k),
                two_s * (j * k - i * r),
                two_s * (i * k - j * r),
                two_s * (j * k + i * r),
                1 - two_s * (i * i + j * j),
            ),
            -1,
        )
        return o.reshape(quaternions.shape[:-1] + (3, 3))

    def _axis_angle_rotation(axis, angle):
        cos, sin = torch.cos(angle), torch.sin(angle)
        one, zero = torch.ones_like(angle), torch.zeros_like(angle)
        if axis == "X":
            r = (one, zero, zero, zero, cos, -sin, zero, sin, cos)
        elif axis == "Y":
            r = (cos, zero, sin, zero, one, zero, -sin, zero, cos)
        else:
            r = (cos, -sin, zero, sin, cos, zero, zero, zero, one)
        return torch.stack(r, -1).reshape(angle.shape + (3, 3))

    def euler_angles_to_matrix(euler_angles, convention="XYZ"):
        mats = [_axis_angle_rotation(ax, ang) for ax, ang in zip(convention, torch.unbind(euler_angles, -1))]
        return mats[0] @ mats[1] @ mats[2]

    def axis_angle_to_matrix(axis_angle):
        angle = torch.linalg.norm(axis_angle, dim=-1, keepdim=True).clamp(min=1e-8)
        axis = axis_angle / angle
        x, y, z = axis.unbind(-1)
        c, s = torch.cos(angle.squeeze(-1)), torch.sin(angle.squeeze(-1))
        C = 1 - c
        R = torch.stack(
            (
                x * x * C + c,
                x * y * C - z * s,
                x * z * C + y * s,
                y * x * C + z * s,
                y * y * C + c,
                y * z * C - x * s,
                z * x * C - y * s,
                z * y * C + x * s,
                z * z * C + c,
            ),
            -1,
        )
        return R.reshape(axis_angle.shape[:-1] + (3, 3))

    def _copysign(a, b):
        return torch.where((a < 0) != (b < 0), -a, a)

    def so3_relative_angle(*args, **kwargs):
        raise RuntimeError("so3_relative_angle is only used in training")

    p3d = types.ModuleType("pytorch3d")
    transforms = types.ModuleType("pytorch3d.transforms")
    rotc = types.ModuleType("pytorch3d.transforms.rotation_conversions")
    so3 = types.ModuleType("pytorch3d.transforms.so3")
    transforms.rotation_6d_to_matrix = rotation_6d_to_matrix
    transforms.quaternion_to_matrix = quaternion_to_matrix
    transforms.euler_angles_to_matrix = euler_angles_to_matrix
    transforms.axis_angle_to_matrix = axis_angle_to_matrix
    rotc._copysign = _copysign
    so3.so3_relative_angle = so3_relative_angle
    sys.modules["pytorch3d"] = p3d
    sys.modules["pytorch3d.transforms"] = transforms
    sys.modules["pytorch3d.transforms.rotation_conversions"] = rotc
    sys.modules["pytorch3d.transforms.so3"] = so3
    p3d.transforms = transforms
    print("Installed lightweight pytorch3d transform stub (inference only)")


def _patch_omni3d(root):
    """Comment out pytorch3d renderer imports and stub names used at import time."""
    math_util = Path(root) / "cubercnn" / "util" / "math_util.py"
    text = math_util.read_text()
    if "WORKSHOP_PATCH" not in text:
        lines = ["# WORKSHOP_PATCH\n"]
        commenting = False
        for line in text.splitlines(True):
            hit = ("pytorch3d.renderer" in line) or ("from pytorch3d.structures" in line)
            if hit or commenting:
                lines.append("# " + line)
                commenting = "(" in line and ")" not in line if hit else (commenting and ")" not in line)
                continue
            lines.append(line)
        text = "".join(lines)
        print("Commented pytorch3d renderer imports")

    stubs = (
        "# WORKSHOP_PATCH_STUBS\n"
        "class _P3DDummy:\n"
        "    def __init__(self, *args, **kwargs):\n"
        "        pass\n"
        "MR = _P3DDummy\n"
        "PointLights = SoftPhongShader = Meshes = TexturesVertex = _P3DDummy\n"
        "PerspectiveCameras = RasterizationSettings = MeshRasterizer = SoftSilhouetteShader = _P3DDummy\n"
    )
    if "WORKSHOP_PATCH_STUBS" not in text:
        # Insert after the first line so class MeshRenderer(MR) can import.
        parts = text.split("\n", 1)
        text = parts[0] + "\n" + stubs + (parts[1] if len(parts) > 1 else "")
        print("Inserted pytorch3d renderer stubs (defines MR)")

    math_util.write_text(text)

    vis_init = Path(root) / "cubercnn" / "vis" / "__init__.py"
    if vis_init.exists() and "WORKSHOP_PATCH" not in vis_init.read_text():
        vis_init.write_text("# WORKSHOP_PATCH\n# skip renderer-based vis\n")
        print("Patched cubercnn/vis/__init__.py")

    rcnn = Path(root) / "cubercnn" / "modeling" / "meta_arch" / "rcnn3d.py"
    rtxt = rcnn.read_text()
    if "from cubercnn import util, vis" in rtxt:
        rcnn.write_text(rtxt.replace("from cubercnn import util, vis", "from cubercnn import util; vis = None"))
        print("Patched rcnn3d.py to skip vis import")

    # Drop cached cubercnn modules so a re-run of this cell picks up the new files.
    for name in list(sys.modules):
        if name == "cubercnn" or name.startswith("cubercnn."):
            del sys.modules[name]


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

_install_pytorch3d_stub()
_patch_omni3d("omni3d")
print("Omni3D patches applied")
print("If you already hit NameError: MR, re-run this cell, then the helper-import cell.")


## Troubleshooting

| Problem | Fix |
|---|---|
| CUDA is unavailable | `Runtime → Change runtime type → T4 GPU`, then restart and Run all |
| Package import fails | `Runtime → Restart session`, then Run all |
| Checkpoint download fails | Re-run the setup / model-load cell |
| Out of memory | Use the smaller default model, or a smaller image |
| A student cell has `???` | That is expected. Fill it in, or set `RUN_STUDENT_ASSEMBLY = False` to skip it |

Do not spend workshop time debugging package conflicts. Restart and Run all first.


If Detectron2 compilation fails, restart the session and run setup again. CPU inference works but is slow.

If you see `NameError: name 'MR' is not defined`, re-run the Omni3D **patch** cell, then the helper-import cell. That error means an older patch commented out PyTorch3D without defining a stub for `MR`.


## 3. Imports


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

import urllib.error
import urllib.request
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

SAMPLE_IMAGES = {
    "bus": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg",
    "zidane": "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg",
    "cats": "http://images.cocodataset.org/val2017/000000039769.jpg",
    "living_room": "http://images.cocodataset.org/val2017/000000000139.jpg",
    "street": "http://images.cocodataset.org/val2017/000000037777.jpg",
}


def download_image(url, path="sample.jpg"):
    path = Path(path)
    if path.exists():
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "object-detection-workshop/1.0"})
    try:
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    except urllib.error.HTTPError as err:
        loc = err.headers.get("Location")
        if err.code in (301, 302, 303, 307, 308) and loc:
            return download_image(loc, path)
        raise
    return path


def load_image(path):
    """Load an RGB uint8 image as a NumPy array (H, W, 3)."""
    return np.array(Image.open(path).convert("RGB"))


def _class_color(cls_id):
    rng = np.random.RandomState(int(cls_id) * 17 + 3)
    return rng.randint(40, 230, size=3) / 255.0


def visualize_detections(
    image,
    boxes,
    scores=None,
    labels=None,
    names=None,
    title=None,
    max_dets=60,
    prompt=None,
):
    """Draw xyxy boxes. `names` maps class id → string."""
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(image)
    ax.axis("off")
    header = title or ""
    if prompt:
        header = (header + "  |  prompt: " + str(prompt)).strip(" |")
    if header:
        ax.set_title(header, fontsize=12)

    boxes = [] if boxes is None else list(boxes)[:max_dets]
    scores = [None] * len(boxes) if scores is None else list(scores)[:max_dets]
    labels = [None] * len(boxes) if labels is None else list(labels)[:max_dets]

    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = [float(v) for v in box]
        cls_id = 0 if label is None else int(label)
        color = _class_color(cls_id)
        ax.add_patch(
            patches.Rectangle(
                (x1, y1),
                max(x2 - x1, 1.0),
                max(y2 - y1, 1.0),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
        )
        name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else (str(label) if label is not None else "")
        caption = name if score is None else f"{name} {float(score):.2f}"
        ax.text(
            x1,
            max(y1 - 4, 12),
            caption,
            color="white",
            fontsize=9,
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85),
        )
    fig.tight_layout()
    plt.show()
    return fig


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import transforms as T

from cubercnn.config import get_cfg_defaults
from cubercnn.modeling.backbone import build_dla_from_vision_fpn_backbone  # noqa: F401
from cubercnn.modeling.meta_arch import build_model
from cubercnn.modeling.proposal_generator import RPNWithIgnore  # noqa: F401
from cubercnn.modeling.roi_heads import ROIHeads3D  # noqa: F401
from cubercnn import util

CONFIG_URL = "cubercnn://omni3d/cubercnn_DLA34_FPN.yaml"
WEIGHTS_URL = "cubercnn://omni3d/cubercnn_DLA34_FPN.pth"


def _local(url):
    if url.startswith(util.CubeRCNNHandler.PREFIX):
        return util.CubeRCNNHandler._get_local_path(util.CubeRCNNHandler, url)
    return url


cfg = get_cfg()
get_cfg_defaults(cfg)
cfg.merge_from_file(_local(CONFIG_URL))
cfg.MODEL.WEIGHTS = WEIGHTS_URL
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
cfg.freeze()

cube_model = build_model(cfg)
DetectionCheckpointer(cube_model).resume_or_load(_local(WEIGHTS_URL), resume=False)
cube_model.eval()
print("Cube R-CNN loaded on", cfg.MODEL.DEVICE)

try:
    CAT_JSON = _local("cubercnn://omni3d/category_meta.json")
    CUBE_NAMES = {i: n for i, n in enumerate(util.load_json(CAT_JSON)["thing_classes"])}
except Exception as exc:
    print("category_meta.json fallback:", exc)
    names = list(getattr(cfg.DATASETS, "CATEGORY_NAMES", []))
    CUBE_NAMES = {i: n for i, n in enumerate(names)}
print("categories:", len(CUBE_NAMES))


def prepare_camera_parameters(image, focal_ndc=4.0, focal_px=None):
    """Build a simple pinhole camera. If EXIF is unknown we assume a centered principal point.

    focal_ndc=4.0 is Omni3D's default when intrinsics are unknown.
    Physical focal length in pixels is roughly focal_ndc * height / 2.
    """
    h, w = image.shape[:2]
    f = float(focal_px) if focal_px else float(focal_ndc) * h / 2.0
    px, py = w / 2.0, h / 2.0
    K = np.array([[f, 0.0, px], [0.0, f, py], [0.0, 0.0, 1.0]], dtype=np.float32)
    return K, {"height": h, "width": w, "focal_px": f, "focal_ndc": f * 2.0 / h}


def _bgr(image_rgb):
    return image_rgb[:, :, ::-1].copy()


@torch.no_grad()
def _forward(image_rgb, K):
    im = _bgr(image_rgb)
    h, w = im.shape[:2]
    aug = T.AugmentationList([T.ResizeShortestEdge(cfg.INPUT.MIN_SIZE_TEST, cfg.INPUT.MAX_SIZE_TEST, "choice")])
    aug_input = T.AugInput(im)
    aug(aug_input)
    image = aug_input.image
    batched = [{
        "image": torch.as_tensor(np.ascontiguousarray(image.transpose(2, 0, 1))).to(cube_model.device),
        "height": h,
        "width": w,
        "K": K,
    }]
    return cube_model(batched)[0]["instances"]


def extract_image_features(image, K):
    """Educational wrapper around Cube R-CNN's 2D visual backbone (runs with the full model)."""
    return {"image": image, "K": K}


def detect_2d_objects(image, K, score_thresh=0.25):
    instances = _forward(image, K)
    keep_t = instances.scores >= float(score_thresh)
    instances = instances[keep_t]
    if len(instances) == 0:
        return {
            "boxes": np.zeros((0, 4), dtype=np.float32),
            "scores": np.zeros((0,), dtype=np.float32),
            "labels": np.zeros((0,), dtype=np.int64),
            "names": CUBE_NAMES,
            "instances": instances,
        }
    boxes = instances.pred_boxes.tensor.detach().cpu().numpy() if instances.has("pred_boxes") else np.zeros((0, 4))
    scores = instances.scores.detach().cpu().numpy()
    labels = instances.pred_classes.detach().cpu().numpy()
    return {
        "boxes": boxes,
        "scores": scores,
        "labels": labels,
        "names": CUBE_NAMES,
        "instances": instances,
    }


def predict_3d_parameters(det2d):
    """Read 3D center, dimensions, and orientation predicted by the cube head."""
    inst = det2d["instances"]
    if len(inst) == 0:
        return {
            "center": np.zeros((0, 3)),
            "dims": np.zeros((0, 3)),
            "pose": np.zeros((0, 3, 3)),
            "corners": np.zeros((0, 8, 3)),
        }
    return {
        "center": inst.pred_center_cam.detach().cpu().numpy(),
        "dims": inst.pred_dimensions.detach().cpu().numpy(),
        "pose": inst.pred_pose.detach().cpu().numpy(),
        "corners": inst.pred_bbox3D.detach().cpu().numpy(),
    }


def decode_3d_boxes(params):
    return params["corners"]


def project_3d_box(corners, K):
    """Project 8 camera-space corners to the image with the pinhole model."""
    corners = np.asarray(corners, dtype=np.float32)
    if corners.ndim == 2:
        corners = corners[None]
    uv = []
    for c in corners:
        x = K @ c.T
        x = x[:2] / np.clip(x[2:3], 1e-6, None)
        uv.append(x.T)
    return np.stack(uv, 0) if uv else np.zeros((0, 8, 2))


CUBE_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 0),
    (4, 5), (5, 6), (6, 7), (7, 4),
    (0, 4), (1, 5), (2, 6), (3, 7),
]


def visualize_3d_detection(image, det2d, params, K, title=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(image)
    axes[0].axis("off")
    axes[0].set_title("2D boxes")
    for box, score, lab in zip(det2d["boxes"], det2d["scores"], det2d["labels"]):
        x1, y1, x2, y2 = box
        color = _class_color(int(lab))
        axes[0].add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, lw=2))
        axes[0].text(x1, max(y1 - 4, 12), f"{CUBE_NAMES.get(int(lab), lab)} {score:.2f}", color="white",
                     fontsize=8, bbox=dict(facecolor=color, edgecolor="none", pad=1, alpha=0.85))

    axes[1].imshow(image)
    axes[1].axis("off")
    axes[1].set_title("3D cuboids (projected)")
    uv = project_3d_box(params["corners"], K)
    for i, poly in enumerate(uv):
        color = _class_color(int(det2d["labels"][i]))
        for a, b in CUBE_EDGES:
            p1, p2 = poly[a], poly[b]
            axes[1].plot([p1[0], p2[0]], [p1[1], p2[1]], color=color, lw=2)
        z = params["center"][i, 2]
        axes[1].text(poly[0, 0], poly[0, 1], f"z={z:.1f}m", color="white", fontsize=8,
                     bbox=dict(facecolor=color, edgecolor="none", pad=1, alpha=0.85))
    if title:
        fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()
    return fig


def run_full_pipeline(image, focal_ndc=4.0, score_thresh=0.25):
    """Black-box demo. For the assembly exercise, call the named stages yourself."""
    K, cam = prepare_camera_parameters(image, focal_ndc=focal_ndc)
    features = extract_image_features(image, K)
    det2d = detect_2d_objects(image, K, score_thresh=score_thresh)
    params = predict_3d_parameters(det2d)
    corners = decode_3d_boxes(params)
    uv = project_3d_box(corners, K)
    return {
        "K": K,
        "cam": cam,
        "features": features,
        "det2d": det2d,
        "params": params,
        "corners": corners,
        "uv": uv,
    }


print("Cube R-CNN helpers ready.")


## 4. Load an example image

An outdoor street scene is a good start. Indoor furniture also works because Omni3D mixed both.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

IMAGE_PATH = download_image(SAMPLE_IMAGES["street"], "street.jpg")
image = load_image(IMAGE_PATH)
print("image:", image.shape)
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.axis("off"); plt.title("Input RGB (no depth)"); plt.show()


## 5–6. The Cube R-CNN pipeline

```text
RGB image
  → 2D visual features
  → 2D object detection
  → 3D parameter prediction   (center, dimensions, orientation)
  → 3D bounding cuboid
  → project cuboid into the image
```

Camera geometry is an input, not something the network can invent perfectly. If the focal length is wrong, metric depth/scale will be wrong even if the cuboid *looks* reasonable on the photo.


## Instructor demo

This cell only calls a provided **one-liner** so Run all still shows a picture. Assemble the named stages yourself below — do not copy `run_full_pipeline`.


In [ ]:
# ==========================================
# INSTRUCTOR PROVIDED — DO NOT MODIFY
# ==========================================

FOCAL_NDC = 4.0  # Omni3D default when true intrinsics are unknown
DEMO = run_full_pipeline(image, focal_ndc=FOCAL_NDC)
print("focal_px", round(DEMO["cam"]["focal_px"], 1))
print("objects:", len(DEMO["det2d"]["boxes"]))
visualize_3d_detection(
    image, DEMO["det2d"], DEMO["params"], DEMO["K"],
    title=f"Cube R-CNN demo | focal_ndc={FOCAL_NDC}",
)


## 7. Student assembly


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================
# Set True after you replace every ??? with the correct function call.
RUN_STUDENT_ASSEMBLY = False

if RUN_STUDENT_ASSEMBLY:
    FOCAL_NDC = 4.0
    K, cam = prepare_camera_parameters(image, focal_ndc=FOCAL_NDC)
    features = ???       # extract_image_features
    det2d = ???          # detect_2d_objects  (pass score_thresh=0.25)
    params = ???         # predict_3d_parameters
    corners = decode_3d_boxes(params)
    _ = project_3d_box(corners, K)
    visualize_3d_detection(image, det2d, params, K, title="Student assembly")
    print("2D/3D objects:", len(det2d["boxes"]))
    print("camera focal (px):", round(cam["focal_px"], 1))
else:
    print('Skipping student assembly. The instructor demo above already ran the pipeline.')
    print('During the exercise: fill in the TODOs, then set RUN_STUDENT_ASSEMBLY = True.')


## 8–9. Experiments

Change focal length, then confidence. Watch estimated depth `z`.


In [ ]:
# ==========================================
# STUDENT TASK
# ==========================================

SCORE = 0.25
for focal_ndc in [2.0, 4.0, 8.0]:
    K_i, cam_i = prepare_camera_parameters(image, focal_ndc=focal_ndc)
    det_i = detect_2d_objects(image, K_i, score_thresh=SCORE)
    par_i = predict_3d_parameters(det_i)
    depths = par_i["center"][:, 2] if len(par_i["center"]) else np.array([])
    print(f"focal_ndc={focal_ndc:.1f}  focal_px={cam_i['focal_px']:.0f}  n={len(det_i['boxes'])}  mean z={np.mean(depths) if len(depths) else float('nan'):.2f}m")
    visualize_3d_detection(image, det_i, par_i, K_i, title=f"focal_ndc={focal_ndc}")


## 10. Think about it

1. What extra information must be predicted to turn a 2D box into a 3D box?
2. What information is **fundamentally unavailable** from a single RGB image and must be inferred?
3. If you double the focal length, do objects tend to look closer, farther, or unchanged on the image — and what happens to metric depth?


## 11. Optional challenge

Run the same pipeline on `SAMPLE_IMAGES["living_room"]`. Indoor objects (chairs, tables) are in Omni3D’s vocabulary too.


## 12. Summary and workshop comparison

Cube R-CNN extends 2D detection with **monocular 3D geometry**. It still needs a camera model.

| Model | Architecture | Input | Output | Main idea |
|---|---|---|---|---|
| YOLO | CNN / one-stage | RGB | 2D boxes | Dense detection + NMS |
| Faster R-CNN | CNN / two-stage | RGB | 2D boxes | Region proposals, then classify/refine |
| RT-DETR | CNN + Transformer | RGB | 2D boxes | Object queries / set prediction |
| YOLO-World | Vision + language | RGB + text | 2D boxes | Open vocabulary |
| Cube R-CNN | Monocular 3D | RGB (+ camera) | 3D boxes | 3D geometry from one image |

```text
                 OBJECT DETECTION
                       │
       ┌───────────────┼────────────────┐
       │               │                │
    CNN-based      Transformer      Vision-Language
       │               │                │
    YOLO / FRCNN    RT-DETR         YOLO-World
       │
       └──────────────→ 3D
                         │
                    Cube R-CNN
```
